In [14]:
import numpy as np
import matplotlib.pyplot as plt
import math
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import re
from typing import List, Dict
from sentence_transformers import SentenceTransformer

import torch
print(torch.cuda.is_available()) 

model = SentenceTransformer("multi-qa-mpnet-base-cos-v1")

False


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
def split_conversations(document: str) -> List[str]:
    parts = re.split(r"(?=Conversation\s+\d+)", document.strip())
    return [part.strip() for part in parts if part.strip()]

In [16]:
# this is used for new line splitter
def lang_chunk(document, chunk_size, chunk_overlap):

    #text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    text_splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    
    texts = text_splitter.split_text(document)

    
    return texts

In [17]:
def get_parts(text: str):
    text = text.replace("\\n", " ")

    text = re.sub(r"Conversation\s+\d+\s*", "", text)

    parts = re.split(r"(User:|Secretary:)", text)

    dialogues = []
    current_speaker = None

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part == "User:":
            current_speaker = "user"
        elif part == "Secretary:":
            current_speaker = "secretary"
        else:
            dialogues.append((current_speaker, " ".join(part.split())))

    return dialogues

In [18]:
def sliding_window_chunk(text: str, window_size: int, stride: int):
    parts = get_parts(text)
    window_size = 4
    stride = 1
    sliding_window = []
    for i in range(0, len(parts) - window_size + 1, stride):
        window = parts[i:i + window_size]

        chunk = "\n".join(f"{speaker}: {text}" for speaker, text in window)
        sliding_window.append(chunk)
    return sliding_window

In [19]:
def pair_dialogues(text: str):
    dialogues = get_parts(text)
    
    pairs = []
    i = 0
    while i < len(dialogues) - 1:
        if dialogues[i][0] == "user" and dialogues[i+1][0] == "secretary":
            pairs.append(
                f"user: {dialogues[i][1]}\nsecretary: {dialogues[i+1][1]}"
            )
            i += 2
        else:
            i += 1

    return pairs

In [20]:
with open("../data/secretary_conversations_dataset.txt", "r") as file:
    content = file.read()
converse = split_conversations(content)


dialogues = []
for conv in converse:
    dialogues.extend(pair_dialogues(conv))

window = []

for conv in converse:
    window.extend(sliding_window_chunk(conv, 3, 1))


combined = dialogues + window
print(f"combined: {combined}")


combined: ['user: Move my meeting with Emma.\nsecretary: Which meeting would you like to move?', 'user: The one at 1 PM Friday.\nsecretary: What time would you like to move it to?', 'user: 10 AM.\nsecretary: Okay. I moved your meeting with Emma to 10 AM on Friday.', 'user: Remind me to send the client report.\nsecretary: When would you like the reminder?', 'user: 4 PM.\nsecretary: Okay. I will remind you at 4 PM to send the client report.', 'user: Schedule a meeting with Olivia.\nsecretary: Sure. What day would you like to schedule it?', 'user: Thursday.\nsecretary: What time works best?', 'user: 2 PM.\nsecretary: Okay. I scheduled a meeting with Olivia on Thursday at 2 PM.', 'user: Schedule a meeting with the marketing team.\nsecretary: What day should I schedule it?', 'user: Friday.\nsecretary: What time should the meeting be?', 'user: 3 PM.\nsecretary: Okay. I scheduled a meeting with the marketing team on Friday at 3 PM.', 'user: Schedule a meeting with Sophia.\nsecretary: Sure. Wh

In [21]:
#embedding
combined_embed = model.encode(combined)
combined_embed

array([[ 0.03887331,  0.01074872, -0.00988112, ...,  0.02076559,
        -0.03460404,  0.004025  ],
       [ 0.02167055,  0.049156  ,  0.00211848, ..., -0.00450141,
        -0.03145745, -0.00900046],
       [ 0.01895726,  0.00063955, -0.01119671, ...,  0.01919769,
        -0.03755363, -0.02392492],
       ...,
       [ 0.03348663,  0.01884465, -0.00277455, ..., -0.00317468,
        -0.01460937, -0.00012606],
       [-0.01224938,  0.04700313, -0.00334877, ...,  0.0066224 ,
        -0.02255001, -0.00442715],
       [ 0.0072197 , -0.01751265,  0.00859157, ..., -0.00468198,
         0.00064696,  0.00151519]], shape=(1548, 768), dtype=float32)

In [22]:
test = ['user: Move my meeting with Emma.\nsecretary: Which meeting would you like to move?', 'user: The one at 1 PM Friday.\nsecretary: What time would you like to move it to?', 'user: 10 AM.\nsecretary: Okay. I moved your meeting with Emma to 10 AM on Friday.', 'user: Remind me to send the client report.\nsecretary: When would you like the reminder?']

user_query = ['can you move my meeting']

test_encode = model.encode(test)
user_encode = model.encode(user_query)

model.similarity(test_encode, user_encode)

tensor([[0.7292],
        [0.5183],
        [0.5835],
        [0.3064]])

: 